# Kalman Historical V2 Analysis\n\n기존 Historical run `20260913_042850`을 재실행하지 않고 분석합니다.\n\n- Qlib experiment recording\n- vectorbt independent validation\n- Equal Weight / Inverse Vol / HRP / Risk Parity / CVaR / CDaR benchmark\n- Walk-forward OOS permutation feature contribution\n- immutable artifact provenance + pinned Git SHA\n\n**Safety:** Research only / Toss execution OFF / Neon write OFF\n

In [ ]:
from google.colab import drive
import json
import shutil
import subprocess
import traceback
from datetime import datetime
from pathlib import Path

PINNED_SHA = "921c70ed66f8a73ecd655637e39234ea2e7d75e4"
RUN_TAG = "20260913_042850"

drive.mount('/content/drive', force_remount=False)

diag_dir = Path('/content/drive/MyDrive/Kalman_Diagnostics')
diag_dir.mkdir(parents=True, exist_ok=True)
bootstrap_status = diag_dir / 'historical_v2_bootstrap_status.json'

def write_bootstrap(status, **extra):
    payload = {
        'status': status,
        'updated_at': datetime.now().astimezone().isoformat(),
        'run_tag': RUN_TAG,
        'pinned_sha': PINNED_SHA,
        **extra,
    }
    tmp = bootstrap_status.with_suffix('.json.tmp')
    tmp.write_text(json.dumps(payload, ensure_ascii=False, indent=2, default=str) + '\n', encoding='utf-8')
    tmp.replace(bootstrap_status)

write_bootstrap('RUNNING', phase='CLONE')

try:
    repo = Path('/content/Codex')
    if repo.exists():
        shutil.rmtree(repo)

    # Use a normal branch clone, then checkout the pinned reachable commit.
    # This is more reliable in Colab than fetching an arbitrary SHA directly.
    subprocess.run(
        ['git', 'clone', '--branch', 'main', 'https://github.com/kimtk94/Codex.git', str(repo)],
        check=True,
    )
    write_bootstrap('RUNNING', phase='PINNED_CHECKOUT')
    subprocess.run(['git', '-C', str(repo), 'checkout', '--detach', PINNED_SHA], check=True)
    checked = subprocess.check_output(['git', '-C', str(repo), 'rev-parse', 'HEAD'], text=True).strip()
    assert checked == PINNED_SHA, (checked, PINNED_SHA)

    gateway = repo / 'kalman-toss-gateway'
    runner = gateway / 'scripts' / 'colab_historical_v2_analysis.py'
    required = [
        runner,
        gateway / 'research' / 'quant_stack' / 'historical_v2_analysis.py',
        gateway / 'research' / 'quant_stack' / 'historical_v2_vectorbt.py',
        gateway / 'research' / 'quant_stack' / 'historical_v2_riskfolio.py',
    ]
    missing = [str(p) for p in required if not p.exists()]
    assert not missing, missing

    write_bootstrap('RUNNING', phase='PY_COMPILE')
    subprocess.run(['python', '-m', 'py_compile', *[str(p) for p in required]], check=True)

    write_bootstrap('RUNNING', phase='ANALYSIS_RUNNER')
    subprocess.run(
        [
            'python', str(runner),
            '--drive-root', '/content/drive/MyDrive',
            '--run-tag', RUN_TAG,
            '--pinned-code-sha', PINNED_SHA,
            '--permutation-repeats', '3',
        ],
        check=True,
    )

    model_root = Path('/content/drive/MyDrive/Market_Model_V2/historical_quant_2017_v1') / RUN_TAG
    summary = model_root / 'analysis_v2' / 'historical_v2_analysis_summary.json'
    assert summary.exists(), summary
    write_bootstrap('COMPLETE', phase='DONE', summary=str(summary))
    print('\nSUMMARY:', summary)
    print(summary.read_text(encoding='utf-8'))
except Exception as exc:
    write_bootstrap(
        'FAIL',
        phase='BOOTSTRAP_OR_RUNNER',
        error_type=type(exc).__name__,
        error=str(exc),
        traceback=traceback.format_exc(),
    )
    raise
